# Tuned scikit-learn models: a fair comparison

Comparing models with one hand-picked setting each says little: an SVM with a bad `C` or an MLP with a bad
hidden size loses because of the setting, not the model. Here every model gets the same tuning budget and the same
features, and only then are they compared.

**The rule that keeps it honest:** settings are chosen by cross-validation on the training part only. The test
part is used once per model, at the very end.

```
training part (80%) ──► 5-fold CV ──► try N random settings per model, keep the best (by macro-F1)
                                              │
                                              ▼
                               refit the best setting on the whole training part
                                              │
test part (20%) ──────────────────────────────┴──► evaluated once
```

**What gets tuned** (the "room" each model gets):

| Choice | Options | Why it matters |
|---|---|---|
| Word n-grams | 1, 1–2, 1–3 | phrases like "at most" vs single words |
| Character n-grams | 1–3, 2–4, 1–5 | catches `10^5`, `O(n log n)`, identifiers |
| Lowercase | yes / no | is "Return" ≠ "return" useful or noise? |
| Feature reduction | all n-grams / keep top 1, 5 or 20% / SVD 100 or 300 | see below |
| Class-weight strength | 0, 0.5, 1 | 1 = fully balanced over-corrected against Medium; 0 = none |
| Model knobs | `C`, `alpha`, hidden sizes, learning rate, regression cut points | the usual |

**Feature reduction, in plain words.** TF-IDF gives up to ~450,000 n-gram columns for ~1,900 problems.
*Keep top k%* (`SelectPercentile`) scores every n-gram by how differently it appears in Easy/Medium/Hard and
keeps the best ones — it uses the labels. *SVD* merges co-occurring n-grams into 100–300 "topic" directions —
it ignores the labels. *All n-grams* is the classic setup for linear models. Instead of arguing which is better,
it's a hyperparameter and the search decides; section 5 shows what it found.

Random search (not a full grid): with many knobs, N random combinations usually get close to the best one much
faster than trying every combination.

In [1]:
import json
import os
import random
import time
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import loguniform
from sklearn.decomposition import TruncatedSVD
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.linear_model import Perceptron
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC

from config import CFG, set_seed
from data.dataset import LABELS, load_split, report
from sklearn_models import OrdinalRegressor, WeightedClassifier, qwk_scorer, text_features
from tracking import log_classification, log_image, log_table, start_run

SEED = set_seed()  # CFG.seed from config.py (override with SEED=... in .env)

# --- tuning budget (model-specific, so it lives here and not in config.py) -------------------
N_ITER = 30     # random settings tried per model; raise to 60-100 for more room, 5 for a quick look
N_JOBS = -1     # parallel CV fits (-1 = all CPU cores)
MODELS = ["perceptron", "lsvm", "svm", "mlp_c", "mlp_r"]

# silence convergence / constant-feature warnings (here and in the parallel workers)
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## 1. Data: all problems, no downsampling

`load_split()` returns the shared stratified train/test split (sizes and seed from `config.py`), the same test set
every model in the repo uses. `X, y` = training part (all tuning happens here), `X_test, y_test` = test part.

In [2]:
X, X_test, y, y_test = load_split()
print(f"{len(X)=}, {len(X_test)=}, train class counts: {np.bincount(y)}")

len(X)=1892, len(X_test)=474, train class counts: [516 965 411]


## 2. Search spaces

One pipeline shape for every model: **features → reduction → model**. Every key below is one knob the search can
turn. `clf__estimator__C` means "the `C` of the model inside the class-weight wrapper".

In [3]:
WEIGHT_POWERS = [0.0] if CFG.class_weighting == "none" else [0.0, 0.5, 1.0]

FEATURES = {
    "features__word__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "features__char__ngram_range": [(1, 3), (2, 4), (1, 5)],
    "features__word__lowercase": [True, False],
    "features__char__lowercase": [True, False],
    "clf__weight_power": WEIGHT_POWERS,
}
ALL = ["passthrough"]                                                          # every n-gram
SELECT = [SelectPercentile(f_classif, percentile=p) for p in (1, 5, 20)]       # top k% by label association
SVD = [TruncatedSVD(n_components=n, random_state=SEED) for n in (100, 300)]   # 100/300 "topic" directions

MLP = dict(max_iter=200, early_stopping=True, random_state=SEED)
SPACES = {
    "perceptron": (WeightedClassifier(Perceptron(penalty="l2", early_stopping=True, random_state=SEED)), {
        "reduce": ALL + SELECT + SVD,
        "clf__estimator__alpha": loguniform(1e-6, 1e-2),
    }),
    "lsvm": (WeightedClassifier(LinearSVC(random_state=SEED)), {
        "reduce": ALL + SELECT + SVD,
        "clf__estimator__C": loguniform(1e-3, 1e1),
    }),
    "svm": (WeightedClassifier(SVC(random_state=SEED)), {                       # RBF kernel
        "reduce": SELECT + SVD,
        "clf__estimator__C": loguniform(1e-1, 1e2),
        "clf__estimator__gamma": ["scale", "auto"],
    }),
    "mlp_c": (WeightedClassifier(MLPClassifier(**MLP)), {
        "reduce": SELECT[:2] + SVD,                                             # all n-grams = too many inputs
        "clf__estimator__hidden_layer_sizes": [(16,), (64,), (128,), (64, 32)],
        "clf__estimator__alpha": loguniform(1e-5, 1e-1),
        "clf__estimator__learning_rate_init": loguniform(1e-4, 1e-2),
    }),
    "mlp_r": (OrdinalRegressor(MLPRegressor(**MLP)), {
        "reduce": SELECT[:2] + SVD,
        "clf__estimator__hidden_layer_sizes": [(16,), (64,), (128,), (64, 32)],
        "clf__estimator__alpha": loguniform(1e-5, 1e-1),
        "clf__estimator__learning_rate_init": loguniform(1e-4, 1e-2),
        "clf__low": [0.5, 0.65, 0.75],                                          # below -> Easy
        "clf__high": [1.25, 1.35, 1.5],                                         # above -> Hard
    }),
}

def make_pipeline(clf):
    return Pipeline([("features", text_features()), ("reduce", "passthrough"), ("clf", clf)])

## 3. Run the search

For each model: `N_ITER` random settings × `CFG.cv_folds` folds on the training part, scored by macro-F1 (QWK is
recorded too). That is 5 models × 30 settings × 5 folds = 750 fits; each rebuilds its TF-IDF (2–6 s), so
expect roughly tens of minutes on a laptop. Try `N_ITER = 5` first: the `minutes` column tells you how long a full run takes. The best setting is refit on the whole training part and only then evaluated on the test set.
Every tried setting is saved to `results/sklearn_search_<model>.csv`.

Each model's search is one MLflow run named `sklearn-<model>-tuned-s<seed>` (experiment `.../sklearn`). It stores the search
space, the best setting, CV and test scores, the confusion matrix and a table of **every tried setting**, which you
can explore in the MLflow UI (select the runs → Compare, or open the `tables/search_trials` artifact).

In [4]:
def reducer_label(r):
    if isinstance(r, SelectPercentile):
        return f"top {r.percentile}%"
    if isinstance(r, TruncatedSVD):
        return f"SVD {r.n_components}"
    return "all n-grams"

def trials_frame(name, search):
    """One row per tried setting: its CV scores and readable values of every knob."""
    cvr = search.cv_results_
    df = pd.DataFrame([{k.replace("clf__estimator__", "").replace("clf__", "").replace("features__", ""):
                        (reducer_label(v) if k == "reduce" else v) for k, v in p.items()} for p in cvr["params"]])
    df = df.rename(columns={"reduce": "reduction", "weight_power": "class-weight power",
                            "word__ngram_range": "word n-grams", "char__ngram_range": "char n-grams",
                            "word__lowercase": "lowercase (word)", "char__lowercase": "lowercase (char)"})
    df.insert(0, "model", name)
    df.insert(1, "macro_f1", cvr["mean_test_macro_f1"])
    df.insert(2, "macro_f1_std", cvr["std_test_macro_f1"])
    df.insert(3, "qwk", cvr["mean_test_qwk"])
    return df.astype({c: str for c in df.columns if df[c].dtype == object})

cv = StratifiedKFold(n_splits=CFG.cv_folds, shuffle=True, random_state=SEED)
searches, rows = {}, []

for name in MODELS:
    clf, space = SPACES[name]
    search = RandomizedSearchCV(
        make_pipeline(clf), {**FEATURES, **space}, n_iter=N_ITER, cv=cv,
        scoring={"macro_f1": "f1_macro", "qwk": qwk_scorer}, refit="macro_f1",
        n_jobs=N_JOBS, random_state=SEED, error_score=np.nan)
    run_config = {"model": name, "n_iter": N_ITER, "cv_folds": CFG.cv_folds,
                  "search_space": {k: str(v) for k, v in {**FEATURES, **space}.items()}}

    with start_run("sklearn", name, "tuned", config=run_config, job_type="search") as run:
        t0 = time.time()
        search.fit(X, y)
        searches[name] = search

        i = search.best_index_
        cvr = search.cv_results_
        pred = search.predict(X_test)
        row = {"model": name,
               "cv_macro_f1": cvr["mean_test_macro_f1"][i], "cv_macro_f1_std": cvr["std_test_macro_f1"][i],
               "cv_qwk": cvr["mean_test_qwk"][i],
               **{f"test_{k}": v for k, v in log_classification(run, y_test, pred, prefix="test").items()},
               "minutes": (time.time() - t0) / 60}
        per_class = classification_report(y_test, pred, labels=[0, 1, 2], target_names=LABELS, output_dict=True, zero_division=0)
        row |= {f"test_{l}_recall": per_class[l]["recall"] for l in LABELS}
        rows.append(row)

        run.config.update({"best_params": {k: (reducer_label(v) if k == "reduce" else str(v))
                                           for k, v in search.best_params_.items()}})
        run.summary.update({"cv/macro_f1": row["cv_macro_f1"], "cv/macro_f1_std": row["cv_macro_f1_std"],
                            "cv/qwk": row["cv_qwk"], "minutes": row["minutes"]})
        log_table(run, "search/trials", trials_frame(name, search))

    pd.DataFrame(cvr).drop(columns="params").astype(str).to_csv(CFG.results_dir / f"sklearn_search_{name}.csv", index=False)
    print(f"  best CV macro-F1 {row['cv_macro_f1']:.3f} ± {row['cv_macro_f1_std']:.3f} in {row['minutes']:.1f} min")

best_params = {name: {k: str(v) for k, v in s.best_params_.items()} for name, s in searches.items()}
json.dump(best_params, open(CFG.models_dir / "sklearn_best_params.json", "w"), indent=2)

KeyboardInterrupt: 

## 4. Results

Compare models by the **CV** columns (1,900 problems × 5 folds, much less noisy) and use the **test** columns to
confirm. A model whose test score is far below its CV score was tuned into noise.

In [ ]:
table = pd.DataFrame(rows).set_index("model").sort_values("cv_macro_f1", ascending=False)
table.to_csv(CFG.results_dir / "sklearn_tuned_results.csv", float_format="%.4f")
table.style.format(precision=3)

In [ ]:
for name in table.index:
    print(f"{name:11}", best_params[name])

In [ ]:
fig, axes = plt.subplots(1, len(searches), figsize=(4.2 * len(searches), 4), squeeze=False)
for ax, name in zip(axes[0], table.index):
    cm = confusion_matrix(y_test, searches[name].predict(X_test), labels=[0, 1, 2])   # rows = true
    sns.heatmap(cm, annot=True, fmt="d", cmap="BuGn", xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=ax)
    ax.set_title(f"{name}  (test macro-F1 {table.loc[name, 'test_macro_f1']:.2f})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.tight_layout()
fig.savefig(CFG.figures_dir / "sklearn_tuned_all.png", dpi=CFG.fig_dpi)
plt.show()

## 5. What the search tells us

Every tried setting is a small experiment. Averaging the CV macro-F1 over all settings that share one choice
(e.g. "SVD 100") shows which choices help, independent of the other knobs. Read it as a trend, not a precise
number: each cell averages only a handful of random settings.

In [ ]:
trials = pd.concat([trials_frame(name, s) for name, s in searches.items()], ignore_index=True)

for knob in ["reduction", "class-weight power", "word n-grams", "char n-grams", "lowercase (word)"]:
    print(f"\nMean CV macro-F1 by {knob} (rows) and model (columns); count of settings in brackets")
    mean = trials.pivot_table(index=knob, columns="model", values="macro_f1", aggfunc="mean")
    count = trials.pivot_table(index=knob, columns="model", values="macro_f1", aggfunc="count")
    print((mean.round(3).astype(str) + " (" + count.fillna(0).astype(int).astype(str) + ")").to_string())

One overview run (`sklearn-comparison-s<seed>`) collects the results table, the confusion matrices and every tried setting of every model, so the whole comparison is one run in MLflow.

In [ ]:
with start_run("sklearn", "comparison", config={"models": MODELS, "n_iter": N_ITER}, job_type="summary") as run:
    log_table(run, "results", table.reset_index())
    log_table(run, "trials", trials)
    log_image(run, "confusion_matrices", CFG.figures_dir / "sklearn_tuned_all.png")
    run.summary.update({"best_model_by_cv": table.index[0]})

## 6. Mistakes of the best model

"Best" is chosen by **CV** macro-F1, not by test score (choosing by test score would quietly tune on the test set).

In [ ]:
best_name = table.index[0]
model = searches[best_name].best_estimator_
print("best model by CV:", best_name)

In [ ]:
predicted_easy_was_hard = []
predicted_hard_was_easy = []
for test_example, target in zip(X_test, y_test):
    prediction = model.predict([test_example])
    
    if prediction == 2 and target == 0:
        predicted_hard_was_easy.append(test_example)
        
    if prediction == 0 and target == 2:
        predicted_easy_was_hard.append(test_example)

In [ ]:
print(f"""{random.choice(predicted_easy_was_hard)}""")

In [ ]:
print(f"""{random.choice(predicted_hard_was_easy)}""")

In [ ]:
problems_with_maximum = []
for example, target in zip(X, y):
    if "maximum" in example:
        problems_with_maximum.append(target)
        
problems_with_maximum = Counter(problems_with_maximum)

In [ ]:
difficulties_maximum, counts_maximum = problems_with_maximum.keys(), problems_with_maximum.values()
difficulties_maximum = [LABELS[k] for k in difficulties_maximum]  # labels in the Counter's own order
print(problems_with_maximum)

plt.pie(counts_maximum, labels=difficulties_maximum, colors=[{"Easy": "blue", "Medium": "green", "Hard": "orange"}[d] for d in difficulties_maximum])
plt.title("Distribution of difficulties of problems that contain \"maximum\"")
plt.show()

In [ ]:
problems_with_minimum = []
for example, target in zip(X, y):
    if "minimum" in example:
        problems_with_minimum.append(target)
        
problems_with_minimum = Counter(problems_with_minimum)

In [ ]:
difficulties_minimum, counts_minimum = problems_with_minimum.keys(), problems_with_minimum.values()
difficulties_minimum = [LABELS[k] for k in difficulties_minimum]  # labels in the Counter's own order
print(problems_with_minimum)

plt.pie(counts_minimum, labels=difficulties_minimum, colors=[{"Easy": "blue", "Medium": "green", "Hard": "orange"}[d] for d in difficulties_minimum])
plt.title("Distribution of difficulties of problems that contain \"minimum\"")
plt.show()